# ECG Exploratory Data Analysis

The existing outputs describe the 206-person group list. They are automated EDA results, not validated HRV. Raw data and participant-level records are not distributed in this package.

In [1]:
from pathlib import Path
import json, sys, subprocess
from IPython.display import display, Image, Markdown
cwd = Path.cwd()
ROOT = next(
    (folder for folder in (cwd, *cwd.parents) if (folder / "analysis/ecg/overview.ipynb").is_file()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open the notebook from inside the project folder.")
ECG_DIR = ROOT / "analysis/ecg"
summary = json.loads((ECG_DIR / "results/aggregate_summary.json").read_text())
for key in ["target_ecg_n", "readable_n", "numeric_complete_n", "duration_differs_from_919p8_n", "final_usable_n", "validated_hrv_computed"]:
    print(key, summary[key])


target_ecg_n 206
readable_n 206
numeric_complete_n 206
duration_differs_from_919p8_n 11
final_usable_n None
validated_hrv_computed False


## Existing cohort figures
Duration agreement does not prove fMRI alignment. Amplitude MAD is not HRV. Detector candidates need review.

## Figures (linked, not embedded, to keep the notebook small)

**01_coverage_and_recording_structure**

![01_coverage_and_recording_structure](figures/01_coverage_and_recording_structure.png)

**02_signal_and_detection_diagnostics**

![02_signal_and_detection_diagnostics](figures/02_signal_and_detection_diagnostics.png)

**03_duration_ecdf_and_individual_points**

![03_duration_ecdf_and_individual_points](figures/03_duration_ecdf_and_individual_points.png)

**04_signal_variability_box_points**

![04_signal_variability_box_points](figures/04_signal_variability_box_points.png)

**05_candidate_intervals_and_paired_counts**

![05_candidate_intervals_and_paired_counts](figures/05_candidate_intervals_and_paired_counts.png)

## Methods

In [2]:
print(json.dumps(json.loads((ECG_DIR / "results/methods.json").read_text()), indent=2))
display(Markdown((ECG_DIR / "FINDINGS.md").read_text()))


{
  "population": "Exactly the 206 IDs in the group-provided PSQ/fMRI/ECG list, matched against the 213-record main-session ECG screen",
  "n_target": 206,
  "neurokit_version": "0.3.0",
  "full_record_detection": "neurokit cleaning and peaks, correct_artifacts=False",
  "engzee_comparison": "independently cleaned first 60 s only",
  "review_flags": {
    "nonfinite": "any nonfinite sample",
    "flat": "constant runs >=1 s",
    "extreme_amplitude": "abs(x-median)>20*1.4826*MAD; any run >=1 s",
    "candidate_intervals": "any outside 300-2000 ms or successive ratio outside [0.5,2]",
    "trigger": "median spacing differs from 1.4 s by >0.005 s"
  },
  "flags_are": "Exploratory review prompts, not approved exclusion rules or validated signal-quality classifications",
  "psq_numeric_values_used": false,
  "validated_hrv_computed": false,
  "selection_source": "https://drive.google.com/file/d/1e4C7_iQ0xQuY2K0MJq3wvjnY0yXlvL-R/view",
  "selection_source_modified_time": "2026-11-19T09:49:5

pus# ECG cohort exploratory analysis
Date: 19 November 2026

## Coverage
Target: 206 main-session ECG records selected from a broader screen of 213 (227 main-cohort participants overall).
Processed: 206. Fully read: 206. Automatic detection completed: 206.
Not yet processed: 0. Read/processing errors: 0.
These counts describe automatic screening, not validated ECG suitability.

## What was measured
Full numeric read (and CRC for gzip inputs), sampling frequency, rows, duration, finite values, amplitude MAD and extremes, constant runs, scanner-trigger counts and spacing. NeuroKit cleaning/detection was run on full finite signals; independently cleaned NeuroKit and Engzee pipelines were compared on the first 60 seconds, both without automatic artifact correction. Individual inspection plots and candidate peak arrays are retained under private/.

## Important distinction
No participant was declared finally usable by this script. No normal-beat reference labels were supplied; no validated RMSSD or PSQ associations were calculated. Numeric PSQ values were not used. A comparison between algorithms is not a detection accuracy estimate. Absence of a listed review flag does not prove a clean recording.

## Review flags (not exclusion criteria)
Constant runs >=1 s; nonfinite values; extreme amplitudes beyond 20 x 1.4826 x MAD from the median for >=1 s; candidate intervals outside 300-2000 ms; successive interval ratios outside [0.5, 2]; unexpected duration/trigger counts or median trigger spacing. These exploratory thresholds are documented before outcome analysis but require methodological review before use as eligibility rules.

Exact same-run MNI header verified in 201 requested participants; not verified in 5. This does not prevent ECG-only EDA but requires resolution before a combined fMRI analysis.

Flag counts (overlapping):
- abrupt_candidate_interval_ratio: 197
- sustained_extreme_amplitude: 5
- trigger_count_not_657: 14
- duration_not_919p8s: 11
- candidate_interval_outside_300_2000ms: 59
- detector_empty_first60: 15
- constant_run_ge_1s: 1

## Next required work
Manual review of full traces and prespecified windows, reference beat annotations or reviewed corrections, fixed analysis-segment selection, and same-scan alignment/respiration QC for the physiology test. The previous 12-person visual pilot remains a separate review; this automatic cohort scan does not replace it.

## Sources
- Babayan et al. (2019), LEMON descriptor: https://doi.org/10.1038/sdata.2018.308
- Official ECG files: https://ftp.gwdg.de/pub/misc/MPI-Leipzig_Mind-Brain-Body-LEMON/Continuous_Peripheral_Physiology_During_MRI_MPILMBB_LEMON/MRI_PERPHYS_joined/
- NeuroKit2 documentation: https://neuropsychology.github.io/NeuroKit/
- Prior participant/file inventory and main-session JSON sidecars audited on 19 November 2026.


## Optional local-file reanalysis
Set DATA_DIR to your ECG files with matching JSON sidecars. The portable script does not download data and does not estimate validated HRV. A separate output folder preserves existing results. The new overview plots are not the original demographic panels because no participant demographics are bundled.

In [3]:
DATA_DIR = ROOT / "data/lemon/pilot"
OUTPUT_DIR = ROOT / "data/derived/ecg/notebook"
RUN_ANALYSIS = False
if RUN_ANALYSIS:
    assert DATA_DIR.is_dir(), "Set DATA_DIR to your local ECG folder."
    subprocess.run([sys.executable, str(ECG_DIR / "scripts/ecg_eda.py"), "--data-dir", str(DATA_DIR),
                    "--output", str(OUTPUT_DIR)], check=True)
    overview = OUTPUT_DIR / "ecg_duration_and_mad.png"
    if overview.is_file():
        display(Image(filename=str(overview), width=1000))
